# Adapters génériques

> Petite couche PyTorch générique pour les AAE qui exposent un encodeur.


In [ ]:
#| default_exp adapters


In [ ]:
#| export
"""Generic PyTorch adapters for AAE-like encoders."""


from typing import Any

import torch
from torch import Tensor, nn

from tell_me_why.tensors import select_tensor


def freeze_module(module: nn.Module, trainable: bool = False) -> nn.Module:
    """Toggle gradient updates for every parameter of a module."""

    for parameter in module.parameters():
        parameter.requires_grad = trainable
    return module


class AAEAdapter(nn.Module):
    """Expose a stable `encode` method for an arbitrary AAE implementation."""

    def __init__(
        self,
        aae: nn.Module,
        *,
        encode_method: str = "encode",
        encoder_attr: str | None = None,
        latent_index: int = 0,
    ) -> None:
        super().__init__()
        self.aae = aae
        self.encode_method = encode_method
        self.encoder_attr = encoder_attr
        self.latent_index = latent_index

    def encode(self, inputs: Any, *args: Any, **kwargs: Any) -> Tensor:
        """Return the latent representation produced by the wrapped AAE."""

        if self.encode_method and hasattr(self.aae, self.encode_method):
            encode = getattr(self.aae, self.encode_method)
            if callable(encode):
                return select_tensor(encode(inputs, *args, **kwargs), self.latent_index)

        encoder_attr = self.encoder_attr or "encoder"
        if hasattr(self.aae, encoder_attr):
            encoder = getattr(self.aae, encoder_attr)
            return select_tensor(encoder(inputs, *args, **kwargs), self.latent_index)

        raise AttributeError(
            "Could not find an encoder. Pass `encode_method` or `encoder_attr` "
            "matching your AAE implementation."
        )

    def forward(self, inputs: Any, *args: Any, **kwargs: Any) -> Tensor:
        return self.encode(inputs, *args, **kwargs)


class GraftedAAEClassifier(nn.Module):
    """Combine a generic AAE encoder with a classifier supplied by the developer."""

    def __init__(
        self,
        aae: nn.Module | AAEAdapter,
        classifier: nn.Module,
        *,
        freeze_aae: bool = True,
        adapter_kwargs: dict[str, Any] | None = None,
    ) -> None:
        super().__init__()
        self.aae_adapter = (
            aae
            if isinstance(aae, AAEAdapter)
            else AAEAdapter(aae, **(adapter_kwargs or {}))
        )
        self.classifier = classifier

        if freeze_aae:
            freeze_module(self.aae_adapter, trainable=False)

    def encode(self, inputs: Any, *args: Any, **kwargs: Any) -> Tensor:
        return self.aae_adapter.encode(inputs, *args, **kwargs)

    def forward(
        self,
        inputs: Any,
        *,
        return_latent: bool = False,
        **encode_kwargs: Any,
    ) -> Tensor | tuple[Tensor, Tensor]:
        latents = self.encode(inputs, **encode_kwargs)
        logits = self.classifier(latents)
        if return_latent:
            return logits, latents
        return logits

    @torch.no_grad()
    def predict(self, inputs: Any, **encode_kwargs: Any) -> Tensor:
        """Return class ids from classifier logits."""

        logits = self.forward(inputs, **encode_kwargs)
        if logits.ndim == 1 or logits.shape[-1] == 1:
            return (logits.reshape(-1) > 0).long()
        return logits.argmax(dim=-1)
